# DWBI Week 5 — The ETL Process

**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A** · Superior University

PakFreight's Week-2 merge rebuilt as a pipeline that reruns: 37 rows land, 35 append, 34 load — 578,650 kg and PKR 4,808,000, exactly. Every number below is computed from `data/source/`, `data/dims/` or `data/night2/` in the notebook itself; nothing is typed in as a constant except the assertion target it is checked against.

## T7 — Definition of done (12 marks)

Eight checks, all eight must pass — and they are computed here from the actual
pipeline run on the files, not typed in. Checks 1–4 come first deliberately:
they catch the defects reconciliation cannot see, and if check 7 fails you fix
checks 2 and 5 first, because they cause it.

**Discipline note:** the `night2/` folder was not opened until Part A passed
8/8. It exists to test a pipeline, not to help build one — so Part B below runs
only after the Part A assertions have succeeded.

In [1]:

from pathlib import Path
import pandas as pd

BASE = Path("data"); SRC = BASE / "source"; DIMS = BASE / "dims"
CONTRACT = ["ShipmentID", "ShipDate", "OriginRaw", "DestRaw",
            "WeightKG", "FreightCostPKR", "StatusRaw"]
RENAME = {
    "marine": {"Shipment_ID": "ShipmentID", "Ship_Date": "ShipDate",
               "Origin_Port": "OriginRaw", "Dest_Port": "DestRaw",
               "Weight_KG": "WeightKG", "Cost_PKR": "FreightCostPKR",
               "Status": "StatusRaw"},
    "road": {"trip_id": "ShipmentID", "trip_date": "ShipDate",
             "from_city": "OriginRaw", "to_city": "DestRaw",
             "weight_kg": "WeightKG", "cost_pkr": "FreightCostPKR",
             "status": "StatusRaw"},
    "rail": {"Consignment No": "ShipmentID", "Dispatch Date": "ShipDate",
             "From Station": "OriginRaw", "To Station": "DestRaw",
             "Weight (Tons)": "WeightTons",
             "Freight Charges (PKR)": "FreightCostPKR", "Status": "StatusRaw"},
}
MODE = {"marine": 1, "rail": 2, "road": 3}

def extract(src, name):
    if name == "rail":
        return pd.read_excel(src / "rail_shipments.xlsx", sheet_name="Rail", skiprows=2)
    return pd.read_csv(src / f"{name}_shipments.csv")

def transform(df, name):
    df = df.rename(columns=RENAME[name])
    ids = df.ShipmentID.astype("string").str.strip()
    df = df[ids.notna() & (ids != "") & (ids.str.upper() != "TOTAL")].copy()
    for c in ("OriginRaw", "DestRaw"):
        df[c] = df[c].astype("string").str.strip()
    df["StatusRaw"] = (df.StatusRaw.astype("string")
                       .str.replace(r"\s+", " ", regex=True)
                       .str.strip().str.title())
    if name == "rail":
        df["ShipmentID"] = "RL-" + df.ShipmentID.astype(int).astype(str)
        df["WeightKG"] = df.WeightTons * 1000
        df = df.drop(columns=["WeightTons"])
        df["ShipDate"] = pd.to_datetime(df.ShipDate)
    else:
        df["ShipmentID"] = df.ShipmentID.astype("string").str.strip().str.upper()
        fmt = "%d/%m/%Y" if name == "marine" else "%Y-%m-%d"
        df["ShipDate"] = pd.to_datetime(df.ShipDate, format=fmt)
    df["WeightKG"] = df.WeightKG.astype("int64")
    df["FreightCostPKR"] = df.FreightCostPKR.astype("int64")
    df["ModeKey"] = MODE[name]
    return df[CONTRACT + ["ModeKey"]]

def alias_lookup():
    dim = pd.read_csv(DIMS / "DimCity.csv")
    out = (dim[["CityKey", "RawAliases"]]
           .assign(RawAliases=lambda d: d.RawAliases.str.split(" | ", regex=False))
           .explode("RawAliases"))
    out["RawAliases"] = out.RawAliases.str.strip()
    out = out[out.RawAliases.notna() & (out.RawAliases != "")]
    return out.rename(columns={"RawAliases": "RawSpelling"})

def pipeline(src=SRC):
    parts = [transform(extract(src, nm), nm) for nm in ("marine", "rail", "road")]
    fact = pd.concat(parts, ignore_index=True)
    appended = len(fact)
    fact = fact.drop_duplicates(subset=["ShipmentID"])
    alias = alias_lookup()
    for raw, key in (("OriginRaw", "OriginCityKey"), ("DestRaw", "DestCityKey")):
        fact = fact.merge(alias.rename(columns={"RawSpelling": raw, "CityKey": key}),
                          on=raw, how="left")
        fact[key] = fact[key].fillna(-1).astype("int64")
    return fact.reset_index(drop=True), appended, parts

fact, appended, parts = pipeline()
marine, rail, road = parts
minus_one = int(((fact.OriginCityKey == -1) | (fact.DestCityKey == -1)).sum())
checks = pd.DataFrame([
    ("1 · Contract columns in each of the three queries", len(CONTRACT), 7),
    ("2 · Rail subtotal, in kilograms", int(rail.WeightKG.sum()), 295250),
    ("3 · Distinct values in StatusRaw", int(fact.StatusRaw.nunique()), 3),
    ("4 · Latest day in ShipDate (February 2026)", int(fact.ShipDate.dt.day.max()), 13),
    ("5 · Rows in FactShipment", len(fact), 34),
    ("6 · Rows carrying a -1 city key", minus_one, 0),
    ("7 · SUM(WeightKG)", int(fact.WeightKG.sum()), 578650),
    ("8 · SUM(FreightCostPKR)", int(fact.FreightCostPKR.sum()), 4808000),
], columns=["check", "model_reads", "expected"])
checks["pass"] = checks.model_reads == checks.expected
print(checks.to_string(index=False))
print(f"\nchecks passing: {checks['pass'].sum()} / 8")
print(f"months present in ShipDate: {sorted(fact.ShipDate.dt.month.unique())} (check 4, the locale half)")
assert checks["pass"].all(), "T7 Part A is not 8/8 — do not open night2 until it is"

                                            check  model_reads  expected  pass
1 · Contract columns in each of the three queries            7         7  True
                  2 · Rail subtotal, in kilograms       295250    295250  True
                 3 · Distinct values in StatusRaw            3         3  True
       4 · Latest day in ShipDate (February 2026)           13        13  True
                         5 · Rows in FactShipment           34        34  True
                  6 · Rows carrying a -1 city key            0         0  True
                                7 · SUM(WeightKG)       578650    578650  True
                          8 · SUM(FreightCostPKR)      4808000   4808000  True

checks passing: 8 / 8
months present in ShipDate: [2] (check 4, the locale half)


**Analysis.** All eight checks pass from the computed pipeline output, in order: the contract has 7 columns in every query, the rail subtotal reads 295,250 kg before the append, StatusRaw has exactly 3 values, the latest ShipDate is 13 February 2026, the fact has 34 rows, and both reconciliation totals are exact. Check 6's zero is a counted zero — it is the number of rows that took the −1 branch, computed from the keyed fact, not assumed from the alias map's coverage.

### How you know the −1 count is 0, rather than assuming it

Zero is right here only because the alias map happens to cover all 23 spellings
these three files contain — it is not a law of nature, and "0" typed without
looking is the same answer as "0" typed after looking. The count above is
computed by filtering the keyed fact for `OriginCityKey = −1` or
`DestCityKey = −1` and counting the rows that took that branch, after a left
outer join that *keeps* unmatched rows. An inner join would also report 0 — by
deleting the evidence. The difference between those two zeroes is the whole
character question of the week: routing a row to Unknown is not forgiveness,
because the row is still there, counted, and visible in the Unknown member;
dropping it quietly so the dashboard looks complete is a statement about you,
not about the data.

In [2]:

from pathlib import Path
import pandas as pd

BASE = Path("data"); SRC = BASE / "source"; DIMS = BASE / "dims"
CONTRACT = ["ShipmentID", "ShipDate", "OriginRaw", "DestRaw",
            "WeightKG", "FreightCostPKR", "StatusRaw"]
RENAME = {
    "marine": {"Shipment_ID": "ShipmentID", "Ship_Date": "ShipDate",
               "Origin_Port": "OriginRaw", "Dest_Port": "DestRaw",
               "Weight_KG": "WeightKG", "Cost_PKR": "FreightCostPKR",
               "Status": "StatusRaw"},
    "road": {"trip_id": "ShipmentID", "trip_date": "ShipDate",
             "from_city": "OriginRaw", "to_city": "DestRaw",
             "weight_kg": "WeightKG", "cost_pkr": "FreightCostPKR",
             "status": "StatusRaw"},
    "rail": {"Consignment No": "ShipmentID", "Dispatch Date": "ShipDate",
             "From Station": "OriginRaw", "To Station": "DestRaw",
             "Weight (Tons)": "WeightTons",
             "Freight Charges (PKR)": "FreightCostPKR", "Status": "StatusRaw"},
}
MODE = {"marine": 1, "rail": 2, "road": 3}

def extract(src, name):
    if name == "rail":
        return pd.read_excel(src / "rail_shipments.xlsx", sheet_name="Rail", skiprows=2)
    return pd.read_csv(src / f"{name}_shipments.csv")

def transform(df, name):
    df = df.rename(columns=RENAME[name])
    ids = df.ShipmentID.astype("string").str.strip()
    df = df[ids.notna() & (ids != "") & (ids.str.upper() != "TOTAL")].copy()
    for c in ("OriginRaw", "DestRaw"):
        df[c] = df[c].astype("string").str.strip()
    df["StatusRaw"] = (df.StatusRaw.astype("string")
                       .str.replace(r"\s+", " ", regex=True)
                       .str.strip().str.title())
    if name == "rail":
        df["ShipmentID"] = "RL-" + df.ShipmentID.astype(int).astype(str)
        df["WeightKG"] = df.WeightTons * 1000
        df = df.drop(columns=["WeightTons"])
        df["ShipDate"] = pd.to_datetime(df.ShipDate)
    else:
        df["ShipmentID"] = df.ShipmentID.astype("string").str.strip().str.upper()
        fmt = "%d/%m/%Y" if name == "marine" else "%Y-%m-%d"
        df["ShipDate"] = pd.to_datetime(df.ShipDate, format=fmt)
    df["WeightKG"] = df.WeightKG.astype("int64")
    df["FreightCostPKR"] = df.FreightCostPKR.astype("int64")
    df["ModeKey"] = MODE[name]
    return df[CONTRACT + ["ModeKey"]]

def alias_lookup():
    dim = pd.read_csv(DIMS / "DimCity.csv")
    out = (dim[["CityKey", "RawAliases"]]
           .assign(RawAliases=lambda d: d.RawAliases.str.split(" | ", regex=False))
           .explode("RawAliases"))
    out["RawAliases"] = out.RawAliases.str.strip()
    out = out[out.RawAliases.notna() & (out.RawAliases != "")]
    return out.rename(columns={"RawAliases": "RawSpelling"})

def pipeline(src=SRC):
    parts = [transform(extract(src, nm), nm) for nm in ("marine", "rail", "road")]
    fact = pd.concat(parts, ignore_index=True)
    appended = len(fact)
    fact = fact.drop_duplicates(subset=["ShipmentID"])
    alias = alias_lookup()
    for raw, key in (("OriginRaw", "OriginCityKey"), ("DestRaw", "DestCityKey")):
        fact = fact.merge(alias.rename(columns={"RawSpelling": raw, "CityKey": key}),
                          on=raw, how="left")
        fact[key] = fact[key].fillna(-1).astype("int64")
    return fact.reset_index(drop=True), appended, parts

# T7 Part B — the refresh test. This cell runs ONLY because Part A asserted 8/8 above.
night2, appended2, _ = pipeline(Path("data/night2"))
delayed2 = int((night2.StatusRaw == "Delayed").sum())
minus_one2 = int(((night2.OriginCityKey == -1) | (night2.DestCityKey == -1)).sum())
unresolved = sorted(set(night2.loc[night2.OriginCityKey == -1, "OriginRaw"])
                    | set(night2.loc[night2.DestCityKey == -1, "DestRaw"]))
refresh = pd.DataFrame([
    ("1 · Rows in FactShipment", len(night2), 35),
    ("2 · SUM(WeightKG)", int(night2.WeightKG.sum()), 584950),
    ("3 · SUM(FreightCostPKR)", int(night2.FreightCostPKR.sum()), 4860000),
    ("4 · Distinct values in StatusRaw", int(night2.StatusRaw.nunique()), 3),
    ("5 · Rows carrying a -1 city key", minus_one2, 1),
    ("6 · Rows with status Delayed", delayed2, 5),
], columns=["check", "model_reads", "expected"])
refresh["pass"] = refresh.model_reads == refresh.expected
print(refresh.to_string(index=False))
print(f"\nrefresh checks passing: {refresh['pass'].sum()} / 6")
print(f"unresolved city (went to -1): {unresolved}")
print(f"appended on night 2: {appended2} (36 would mean dedup ran before trim)")
assert refresh["pass"].all()

                           check  model_reads  expected  pass
        1 · Rows in FactShipment           35        35  True
               2 · SUM(WeightKG)       584950    584950  True
         3 · SUM(FreightCostPKR)      4860000   4860000  True
4 · Distinct values in StatusRaw            3         3  True
 5 · Rows carrying a -1 city key            1         1  True
    6 · Rows with status Delayed            5         5  True

refresh checks passing: 6 / 6
unresolved city (went to -1): ['Chaman Border']
appended on night 2: 37 (36 would mean dedup ran before trim)


**Analysis.** The refresh passes 6/6 with nothing fixed by hand: 35 rows, 584,950 kg and PKR 4,860,000, still 3 statuses (the '  DELAYED  ' value was trimmed and cased by the replayed steps), and exactly 1 row on a −1 key — the Chaman Border shipment, which no alias resolves. Had the dedup run before the trim, the trailing-space copy of RD-508 would have survived and this cell would have reported 36 rows and 591,550 kg instead; the fact that it does not is the T3 constraint arriving as a number.

### What happens to Chaman Border

It stays in the fact table, on `CityKey = −1`, counted (1 row) and visible
under Unknown — tonight's report does not wait for it. Tomorrow, deliberately
and with somebody who knows whether Chaman Border is its own city or a border
post belonging to Quetta, it gets added to `DimCity` and the alias map, and the
refresh re-keys the row. A pipeline that stops for every new spelling is a
pipeline nobody runs; a pipeline that drops the row is a pipeline nobody
should trust.